In [1]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [7]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [8]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "mqv:": "https://www.w3.org/2019/wot/mqtt#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [9]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Network Infrastructure

### CQ3.01u - Which Network Participants are exists?

In [5]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?participant ?type WHERE {
  ?participant a mqtt:NetworkParticipant ;
               a ?type .
  ?type rdfs:subClassOf mqtt:NetworkParticipant .

  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


,participant,type
0,ex:Broker_emqx%40172.19.0.3,mqtt:Client
1,ex:Client_Client%201,mqtt:Client
2,ex:Broker_emqx%40172.19.0.3,mqtt:Broker
3,ex:NetworkConnection_%5Bobject%20Object%5D,mqtt:Broker
4,ex:Broker_emqx%40172.19.0.5,mqtt:Broker


### CQ3.02u - Which MQTT Clients are connected to which Broker?

In [6]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?broker WHERE {
  ?client a mqtt:Client ;
          mqtt:isConnectedToBroker ?broker .
  ?broker a mqtt:Broker .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


,client,broker
0,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3
1,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.5


### CQ3.03u - Which MQTT Broker has which connected Client?

In [7]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?client WHERE {
  ?broker a mqtt:Broker ;
          mqtt:hasConnectedClient ?client .
  ?client a mqtt:Client .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short


,broker,client
0,ex:Broker_emqx%40172.19.0.3,ex:Client_Client%201
1,ex:Broker_emqx%40172.19.0.5,ex:Client_Client%201


###  CQ3.04u - Which MQTT participants use which MQTT Network Connections?

In [8]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?participant ?type ?connection WHERE {
  ?participant a ?type ;
               mqtt:usesConnection ?connection .
  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,participant,type,connection
0,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:NetworkConnection_%5Bobject%20Object%5D
1,ex:Client_Client%201,mqtt:Client,ex:NetworkConnection_%5Bobject%20Object%5D
2,ex:Broker_emqx%40172.19.0.3,mqtt:Broker,ex:NetworkConnection_%5Bobject%20Object%5D
3,ex:Broker_emqx%40172.19.0.5,mqtt:Broker,ex:NetworkConnection_%5Bobject%20Object%5D


### CQ3.05u - Which MQTT NetworkConnections exist, are they TLS-secured, and who initiated/accepted them?


In [9]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?connection ?tls ?client ?broker WHERE {
  ?connection a mqtt:NetworkConnection .
  OPTIONAL { ?connection mqtt:usesTLS ?tls }
  OPTIONAL { ?connection mqtt:isInitiatedBy ?client }
  OPTIONAL { ?broker mqtt:isAcceptedBy ?connection }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,connection,tls,client,broker
0,ex:Broker_emqx%40172.19.0.3,NaN,NaN,NaN
1,ex:NetworkConnection_%5Bobject%20Object%5D,NaN,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3
2,ex:NetworkConnection_%5Bobject%20Object%5D,NaN,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.5
3,ex:Broker_emqx%40172.19.0.5,NaN,NaN,NaN


### CQ3.06u - Which MQTT Clients exists under which ClientID?

In [10]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?clientID WHERE {
  ?client a mqtt:Client .
  OPTIONAL { ?client mqtt:hasClientID ?clientID }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,client,clientID
0,ex:Broker_emqx%40172.19.0.3,NaN
1,ex:Client_Client%201,Client 1


### CQ3.07u - Which broker exists under which host and port configuration?

In [11]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?broker ?host ?port WHERE {
  ?broker a mqtt:Broker ;
          OPTIONAL { ?broker mqtt:hasHostAddress ?host }
          OPTIONAL { ?broker mqtt:hasPortNumber ?port }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,broker,host,port
0,ex:Broker_emqx%40172.19.0.3,NaN,NaN
1,ex:NetworkConnection_%5Bobject%20Object%5D,NaN,NaN
2,ex:Broker_emqx%40172.19.0.5,NaN,NaN


### CQ3.08u - Which Control Packets are send by which Network Participant?

In [12]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?participant ?type ?packet WHERE {
  ?participant a ?type ;
             mqtt:sendsControlPacket ?packet .
  ?type rdfs:subClassOf mqtt:NetworkParticipant .
  FILTER(?type IN (mqtt:Client, mqtt:Broker)) .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,participant,type,packet
0,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PublishPacket_1764845861738
1,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:AuthPacket_1764845861737
2,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PubcompPacket_1764845868422
3,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PubackPacket_1764845868422
4,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PubrecPacket_1764845868422
5,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PubrelPacket_1764845868422
6,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:AuthPacket_1764845868422
7,ex:Broker_emqx%40172.19.0.3,mqtt:Client,ex:PublishPacket_1764845868422
8,ex:Client_Client%201,mqtt:Client,ex:UnsubscribePacket_1764800251677
9,ex:Client_Client%201,mqtt:Client,ex:PublishPacket_1764845861738


### CQ3.09u - Which Control Packets are send between both Network Participants?

In [13]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?client ?broker ?packet WHERE {
  ?client a mqtt:Client ;
        mqtt:sendsControlPacket ?packet .
  ?broker a mqtt:Broker ;
        mqtt:sendsControlPacket ?packet .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,client,broker,packet
0,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PublishPacket_1764845861738
1,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:AuthPacket_1764845861737
2,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PubcompPacket_1764845868422
3,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PubackPacket_1764845868422
4,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PubrecPacket_1764845868422
5,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PubrelPacket_1764845868422
6,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:AuthPacket_1764845868422
7,ex:Broker_emqx%40172.19.0.3,ex:Broker_emqx%40172.19.0.3,ex:PublishPacket_1764845868422
8,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3,ex:PublishPacket_1764845861738
9,ex:Client_Client%201,ex:Broker_emqx%40172.19.0.3,ex:AuthPacket_1764845861737


### CQ3.10u - Under which Network Connectiom, which Session is established?

In [10]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?session ?connection WHERE {
  ?session a mqtt:Session ;
        mqtt:establishedByNetworkConnection ?connection .
  ?connection a mqtt:NetworkConnection ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,session,connection


### CQ3.11u - Which Session establishes which Network Connection?

In [11]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?session ?connection WHERE {
  ?connection a mqtt:Connection ;
        mqtt:establishesSession ?session .
  ?session a mqtt:Session ;
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

,session,connection
